# NYC Yellow Taxi Trip Data Analysis

**Dataset:** `yellow_tripdata_2025-11.parquet` (NYC TLC Trip Record Data, November 2025)

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, NLTK

---
## Loading the Dataset

We start by loading the parquet file and checking its structure.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Setup done')

In [ ]:
df = pd.read_parquet('yellow_tripdata_2025-11.parquet')
print(f'Rows: {df.shape[0]:,}  Columns: {df.shape[1]}')

In [ ]:
df.head(10)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
# Quick stats on key numeric columns
print(f'Avg fare: ${np.nanmean(df["fare_amount"]):.2f}')
print(f'Median fare: ${np.nanmedian(df["fare_amount"]):.2f}')
print(f'Std fare: ${np.nanstd(df["fare_amount"]):.2f}')
print(f'Avg trip distance: {np.nanmean(df["trip_distance"]):.2f} miles')
print(f'Max tip: ${np.nanmax(df["tip_amount"]):.2f}')
print(f'Total revenue: ${np.nansum(df["total_amount"]):,.2f}')

In [ ]:
# Check value counts for categorical fields
print('VendorID distribution:')
print(df['VendorID'].value_counts())
print('\nPayment type distribution:')
print(df['payment_type'].value_counts())

The dataset has millions of records with 19 columns covering fares, distances, timestamps and location IDs. Looking at the value counts, one vendor handles the bulk of trips and credit card is the dominant payment type.

---
## Data Cleaning and Visualization

Check for missing values, remove invalid records, then visualize the main distributions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Plotting libs ready')

In [ ]:
# Check missing values
nulls = df.isnull().sum()
nulls_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Missing': nulls, 'Pct': nulls_pct})
null_df[null_df['Missing'] > 0]

In [ ]:
# Clean the data
df_clean = df.copy()

# Drop rows where passenger_count is null or 0
df_clean = df_clean[df_clean['passenger_count'].notna()]
df_clean = df_clean[df_clean['passenger_count'] > 0]

# Remove negative/zero fares and unreasonable values
df_clean = df_clean[df_clean['fare_amount'] > 0]
df_clean = df_clean[df_clean['fare_amount'] < 500]
df_clean = df_clean[df_clean['trip_distance'] > 0]
df_clean = df_clean[df_clean['trip_distance'] < 100]
df_clean = df_clean[df_clean['total_amount'] > 0]

# Remove negative tips
df_clean = df_clean[df_clean['tip_amount'] >= 0]

print(f'Before cleaning: {len(df):,} rows')
print(f'After cleaning: {len(df_clean):,} rows')
print(f'Removed: {len(df) - len(df_clean):,} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')

In [ ]:
# Fare distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['fare_amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Fare Amount Distribution')
axes[0].set_xlabel('Fare ($)')
axes[0].set_ylabel('Count')

axes[1].hist(df_clean['fare_amount'][df_clean['fare_amount'] < 60], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Fare Amount (Zoomed < $60)')
axes[1].set_xlabel('Fare ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numeric columns
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(y=df_clean['trip_distance'][df_clean['trip_distance'] < 30], ax=axes[0], color='lightblue')
axes[0].set_title('Trip Distance')

sns.boxplot(y=df_clean['fare_amount'][df_clean['fare_amount'] < 80], ax=axes[1], color='lightgreen')
axes[1].set_title('Fare Amount')

sns.boxplot(y=df_clean['tip_amount'][df_clean['tip_amount'] < 20], ax=axes[2], color='lightyellow')
axes[2].set_title('Tip Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Payment type bar chart
pay_labels = {0:'Flex Fare', 1:'Credit Card', 2:'Cash', 3:'No Charge', 4:'Dispute', 5:'Unknown', 6:'Voided'}
pay_counts = df_clean['payment_type'].map(pay_labels).value_counts()

plt.figure(figsize=(10, 5))
pay_counts.plot(kind='bar', color=sns.color_palette('Set2', len(pay_counts)), edgecolor='black')
plt.title('Payment Type Distribution')
plt.xlabel('Payment Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['fare_amount','trip_distance','tip_amount','tolls_amount',
    'total_amount','congestion_surcharge','passenger_count','extra','mta_tax']

plt.figure(figsize=(10, 8))
corr = df_clean[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Trips by hour of day
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour

hourly = df_clean['pickup_hour'].value_counts().sort_index()
plt.figure(figsize=(12, 5))
plt.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
plt.title('Number of Trips by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Trip Count')
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# Trips by day of week
df_clean['pickup_day'] = df_clean['tpep_pickup_datetime'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df_clean['pickup_day'].value_counts().reindex(day_order)

plt.figure(figsize=(10, 5))
plt.bar(daily.index, daily.values, color='coral', edgecolor='white')
plt.title('Trips by Day of Week')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

After cleaning we removed a small fraction of bad records (negative fares, zero passengers etc). The fare histogram is clearly right-skewed, most trips cost under $20. Credit card dominates as payment method. The heatmap shows trip_distance and fare_amount are highly correlated, and tip_amount also tracks fare well. Peak taxi usage is during evening rush hours, and weekdays see more trips than weekends.